# LeetCode 3486 — Longest Special Path II bằng Python

- Link: https://leetcode.com/problems/longest-special-path-ii/
- Độ khó: **Hard**
- Chủ đề: **DFS trên cây** + **đường đi gốc → nút như một mảng**, vị trí xuất hiện gần nhất của mỗi giá trị, **giữ hai giá trị lớn nhất** để xác định điểm bắt đầu cửa sổ, **tổng tiền tố** độ dài

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestSpecialPath(self, edges: List[List[int]], nums: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhìn đường đi từ gốc tới nút hiện tại như một **mảng động** (push khi đi xuống, pop khi quay lui) và đưa bài trên cây về bài **cửa sổ trên mảng**.
2. Mô tả điều kiện "mọi giá trị phân biệt, trừ tối đa một giá trị xuất hiện hai lần" bằng **vị trí xuất hiện trước đó** `prev[k]`.
3. Chứng minh điểm bắt đầu tốt nhất là `L = (giá trị lớn thứ hai của prev trên đường) + 1` ⇒ chỉ cần mang theo **hai số** khi DFS.
4. Viết DFS **không đệ quy** để chịu được cây dạng chuỗi `5·10^4` nút.
5. Xử lý tiêu chí phụ: cùng độ dài lớn nhất thì lấy **ít nút nhất**.

## 2. Đề bài & ví dụ

Cho cây vô hướng gốc `0`, `n` nút. `edges[i] = [u, v, length]`. Mỗi nút có giá trị `nums[i]`.

Một **đường đặc biệt** là đường đi **xuống** từ một tổ tiên tới một hậu duệ (có thể chỉ gồm một nút), trong đó mọi giá trị nút phân biệt, **trừ tối đa một giá trị được xuất hiện hai lần**.

Trả về `[độ dài lớn nhất, số nút ít nhất trong các đường đặc biệt dài nhất]`.

**Ví dụ 1:** `edges = [[0,1,1],[1,2,3],[1,3,1],[2,4,6],[4,7,2],[3,5,2],[3,6,5],[6,8,3]]`, `nums = [1,1,0,3,1,2,1,1,0]` → `[9, 3]` (đường `1→2→4` và `1→3→6→8` đều dài 9; ít nút nhất là 3).

**Ví dụ 2:** `edges = [[1,0,3],[0,2,4],[0,3,5]]`, `nums = [1,1,0,2]` → `[5, 2]` (đường `0→3`).

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `1 ≤ length ≤ 10^3`, `0 ≤ nums[i] ≤ 5·10^4`.

In [1]:
from typing import List
from collections import defaultdict, Counter

examples = [
    ([[0,1,1],[1,2,3],[1,3,1],[2,4,6],[4,7,2],[3,5,2],[3,6,5],[6,8,3]], [1,1,0,3,1,2,1,1,0], [9, 3]),
    ([[1,0,3],[0,2,4],[0,3,5]], [1,1,0,2], [5, 2]),
]
for e, nums, exp in examples:
    print('n =', len(nums), '| kỳ vọng', exp)

n = 9 | kỳ vọng [9, 3]
n = 4 | kỳ vọng [5, 2]


## 3. Kiến thức nền

### 3.1. Dựng danh sách kề và tìm cha (BFS từ gốc)

Cây cho dưới dạng cạnh vô hướng; ta cần hướng "xuống" nên BFS từ gốc `0` để biết `parent` và trọng số cạnh lên cha.

In [2]:
def build(n, edges):
    g = [[] for _ in range(n)]
    for u, v, w in edges:
        g[u].append((v, w))
        g[v].append((u, w))
    return g

def parents(n, edges):
    g = build(n, edges)
    par, pw = [-1] * n, [0] * n
    order, seen = [0], [False] * n
    seen[0] = True
    for u in order:
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True
                par[v], pw[v] = u, w
                order.append(v)
    return par, pw, order

e, nums, _ = examples[0]
par, pw, order = parents(len(nums), e)
print('parent:', par)
print('trọng số cạnh lên cha:', pw)

parent: [-1, 0, 1, 1, 2, 3, 3, 4, 6]
trọng số cạnh lên cha: [0, 1, 3, 1, 6, 2, 5, 2, 3]


### 3.2. Tổng tiền tố trên đường đi

Nếu `dist[k]` là khoảng cách từ gốc tới nút thứ `k` trên đường hiện tại thì độ dài đoạn từ vị trí `L` tới `d` là `dist[d] − dist[L]`, số nút là `d − L + 1`.

### 3.3. Mảng `prev` — vị trí xuất hiện trước đó

Trên một mảng `a`, `prev[k]` = vị trí lớn nhất `j < k` với `a[j] = a[k]` (hoặc `−1`). Đoạn `[L, d]`:

- không có giá trị lặp ⇔ không có `k ∈ [L, d]` với `prev[k] ≥ L`;
- "tối đa một giá trị xuất hiện hai lần" ⇔ **có tối đa một** `k ∈ [L, d]` với `prev[k] ≥ L`.

(Một giá trị xuất hiện ba lần ở `a < b < c` cho `prev[b] = a`, `prev[c] = b`, hai chỉ số cùng vi phạm ⇒ bị loại đúng như đề.)

In [3]:
def prev_array(a):
    last, prev = {}, []
    for k, x in enumerate(a):
        prev.append(last.get(x, -1))
        last[x] = k
    return prev

def ok_window(a, L, d):
    prev = prev_array(a)
    return sum(1 for k in range(L, d + 1) if prev[k] >= L) <= 1

a = [1, 1, 0, 1, 2]
print('prev =', prev_array(a))
print('[0..2] hợp lệ?', ok_window(a, 0, 2), '| [0..3] hợp lệ?', ok_window(a, 0, 3), '| [1..4] hợp lệ?', ok_window(a, 1, 4))

prev = [-1, 0, -1, 1, -1]
[0..2] hợp lệ? True | [0..3] hợp lệ? False | [1..4] hợp lệ? True


### 3.4. DFS không đệ quy

Python mặc định giới hạn đệ quy ~1000; cây dạng chuỗi `5·10^4` nút sẽ vỡ. Dùng ngăn xếp tự quản, mỗi nút có hai pha: **vào** (push trạng thái) và **ra** (hoàn tác).

In [4]:
def iterative_preorder(n, edges):
    g = build(n, edges)
    out, stack = [], [(0, -1, 0)]          # (nút, cha, pha)
    while stack:
        u, p, phase = stack.pop()
        if phase == 1:
            out.append(f'ra {u}')
            continue
        out.append(f'vào {u}')
        stack.append((u, p, 1))
        for v, _ in g[u]:
            if v != p:
                stack.append((v, u, 0))
    return out

print(iterative_preorder(4, examples[1][0]))

['vào 0', 'vào 3', 'ra 3', 'vào 2', 'ra 2', 'vào 1', 'ra 1', 'ra 0']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mọi đường đi xuống kết thúc tại nút `x` là một **đoạn cuối** của đường gốc → `x`. Nếu cố định điểm cuối `x`, ta muốn điểm đầu **càng cao càng tốt** (đường càng dài, vì mọi cạnh có độ dài ≥ 1) mà vẫn hợp lệ. Tính chất hợp lệ là **đơn điệu**: nếu `[L, d]` hợp lệ thì `[L+1, d]` cũng hợp lệ. ⇒ Có một `L` nhỏ nhất, và đoạn `[L, d]` là ứng viên duy nhất cần xét tại `x`. Đoạn ngắn hơn chỉ có độ dài nhỏ hơn hẳn, nên tiêu chí "ít nút nhất" chỉ so giữa các điểm cuối khác nhau.

Tính tay ví dụ 1, điểm cuối nút `8`: đường gốc → 8 là `0,1,3,6,8` với giá trị `1,1,3,1,0`. Giá trị 1 xuất hiện 3 lần ⇒ phải bỏ ít nhất lần đầu: đoạn `1,3,6,8` (giá trị `1,3,1,0`) hợp lệ, độ dài `1 + 5 + 3 = 9`, 4 nút. Điểm cuối nút `4`: đường `0,1,2,4` giá trị `1,1,0,1` ⇒ đoạn `1,2,4` (giá trị `1,0,1`), độ dài `3 + 6 = 9`, 3 nút. ⇒ `[9, 3]`.

In [5]:
path_vals = [nums[v] for v in [0, 1, 3, 6, 8]]
print('giá trị đường 0→8:', path_vals, '| prev =', prev_array(path_vals))
print('L nhỏ nhất hợp lệ:', min(L for L in range(5) if ok_window(path_vals, L, 4)))

giá trị đường 0→8: [1, 1, 3, 1, 0] | prev = [-1, 0, -1, 1, -1]
L nhỏ nhất hợp lệ: 1


## 5. Cách 1 — Brute Force: từ mỗi nút đi ngược lên

Với mỗi nút `x`, leo dần lên cha, duy trì `Counter` và số "lần xuất hiện dư". Dừng khi dư > 1. Cập nhật đáp án tại mỗi bước.

In [6]:
def longest_brute(edges, nums):
    n = len(nums)
    par, pw, _ = parents(n, edges)
    best = (0, -1)                        # (độ dài, -số nút) để max
    for x in range(n):
        cnt, length, k, extra, y = Counter(), 0, 0, 0, x
        while y != -1:
            cnt[nums[y]] += 1
            k += 1
            if cnt[nums[y]] >= 2:
                extra += 1
            if extra > 1:
                break
            best = max(best, (length, -k))
            length += pw[y]
            y = par[y]
    return [best[0], -best[1]]

for e, nums_, exp in examples:
    print(longest_brute(e, nums_), '| kỳ vọng', exp)

[9, 3] | kỳ vọng [9, 3]
[5, 2] | kỳ vọng [5, 2]


Lưu ý: `length` cộng `pw[y]` **sau** khi ghi nhận — độ dài đường từ `y` xuống `x` là tổng các cạnh dưới `y`.

**Độ phức tạp:** `O(n · depth)` — cây dạng chuỗi với giá trị phân biệt cho `O(n²)` ≈ `1,25·10^9` bước: quá chậm.

## 6. Ý tưởng thuật toán / bước đột phá

Trên đường gốc → nút hiện tại (độ sâu `d`), gọi `prev[k]` như mục 3.3. Ta cần `L` nhỏ nhất sao cho **tối đa một** `k ∈ [L, d]` có `prev[k] ≥ L`.

**Bổ đề.** Với `k < L` thì `prev[k] < k < L`, nên điều kiện tương đương: tối đa một `k ∈ [0, d]` có `prev[k] ≥ L`. Gọi `M1 ≥ M2` là hai giá trị lớn nhất của `{prev[0..d]}` ⇒ `L* = M2 + 1`.

Vậy khi DFS chỉ cần **mang theo hai số `(M1, M2)`** (truyền xuống con, tự "hoàn tác" khi quay lui vì là tham số), cộng thêm:

- `last[v]`: **ngăn xếp** các độ sâu nơi giá trị `v` xuất hiện trên đường hiện tại (đỉnh ngăn xếp là lần gần nhất);
- `dist`: ngăn xếp tổng tiền tố độ dài.

Tại nút độ sâu `d`: `p = last[v][-1]` (hoặc `−1`), cập nhật `(M1, M2)` với `p`, `L = M2 + 1`, ứng viên `(dist[d] − dist[L], d − L + 1)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cửa sổ trượt trên đường đi, có hoàn tác

Giữ con trỏ trái `L` và `Counter` của cửa sổ, số lần dư `extra`. Vào nút: thêm giá trị, rồi **dịch `L`** tới khi `extra ≤ 1`. Khi quay lui phải **trả lại** các giá trị đã bỏ ra khỏi cửa sổ. Đúng, nhưng việc hoàn tác làm mất tính khấu hao của cửa sổ trượt: trên cây "chổi" (một chuỗi dài rồi nhiều nhánh lá), mỗi lá có thể phải dịch `L` qua cả chuỗi ⇒ tệ nhất `O(n · depth)`.

In [7]:
def longest_window(edges, nums):
    n = len(nums)
    g = build(n, edges)
    path_vals, dist = [], []
    cnt = Counter()
    state = {'L': 0, 'extra': 0}
    best = [0, 1]
    stack = [(0, -1, 0, 0, None)]
    while stack:
        u, p, w, phase, saved = stack.pop()
        if phase == 1:
            old_L, old_extra = saved
            x = path_vals.pop(); dist.pop()
            cnt[x] -= 1
            for k in range(old_L, state['L']):          # trả lại các phần tử đã bị bỏ
                cnt[path_vals[k]] += 1
            state['L'], state['extra'] = old_L, old_extra
            continue
        saved = (state['L'], state['extra'])
        d = len(path_vals)
        dist.append((dist[-1] if dist else 0) + w)
        path_vals.append(nums[u])
        cnt[nums[u]] += 1
        if cnt[nums[u]] >= 2:
            state['extra'] += 1
        while state['extra'] > 1:
            y = path_vals[state['L']]
            if cnt[y] >= 2:
                state['extra'] -= 1
            cnt[y] -= 1
            state['L'] += 1
        L = state['L']
        cand = [dist[d] - dist[L], d - L + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, 1, saved))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, None))
    return best

for e, nums_, exp in examples:
    print(longest_window(e, nums_), '| kỳ vọng', exp)

[9, 3] | kỳ vọng [9, 3]
[5, 2] | kỳ vọng [5, 2]


Chi tiết then chốt khi hoàn tác: `extra` được khôi phục từ bản lưu; `cnt` phải cộng lại đúng các phần tử `path_vals[old_L .. L−1]` đã bị bỏ, và trừ phần tử vừa thêm. Ngoài ra `extra` đếm **số lần xuất hiện dư**, nên giá trị lặp 3 lần tính là 2 ⇒ tự động bị loại.

### 7.2. Cách 3 — Hai giá trị `prev` lớn nhất (lời giải chính, `O(n)`)

In [8]:
def longest_top2(edges, nums):
    n = len(nums)
    g = build(n, edges)
    last = defaultdict(list)            # giá trị -> ngăn xếp độ sâu
    dist = []
    best = [0, 1]
    stack = [(0, -1, 0, 0, -1, -1)]     # (nút, cha, w, pha, M1, M2)
    while stack:
        u, p, w, phase, m1, m2 = stack.pop()
        if phase == 1:
            last[nums[u]].pop()
            dist.pop()
            continue
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w)
        st = last[nums[u]]
        pv = st[-1] if st else -1
        if pv > m1:
            m1, m2 = pv, m1
        elif pv > m2:
            m2 = pv
        st.append(d)
        L = m2 + 1
        length, k = dist[d] - dist[L], d - L + 1
        if length > best[0] or (length == best[0] and k < best[1]):
            best = [length, k]
        stack.append((u, p, w, 1, m1, m2))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, m1, m2))
    return best

for e, nums_, exp in examples:
    print(longest_top2(e, nums_), '| kỳ vọng', exp)

[9, 3] | kỳ vọng [9, 3]
[5, 2] | kỳ vọng [5, 2]


### 7.3. Cách 4 — Đệ quy (gọn, cần nâng giới hạn và chạy trong thread có stack lớn)

Cùng ý tưởng cách 3 nhưng viết đệ quy cho dễ đọc. Để không vỡ với chuỗi dài, chạy trong `threading.Thread` với `threading.stack_size` lớn và `sys.setrecursionlimit`. Trên LeetCode cách này có thể dùng được, nhưng bản lặp (cách 3) an toàn hơn.

In [9]:
import sys, threading

def longest_recursive(edges, nums):
    n = len(nums)
    g = build(n, edges)
    last = defaultdict(list)
    dist = []
    best = [0, 1]

    def dfs(u, p, w, m1, m2):
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w)
        st = last[nums[u]]
        pv = st[-1] if st else -1
        if pv > m1:
            m1, m2 = pv, m1
        elif pv > m2:
            m2 = pv
        st.append(d)
        L = m2 + 1
        cand = [dist[d] - dist[L], d - L + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best[:] = cand
        for v, ww in g[u]:
            if v != p:
                dfs(v, u, ww, m1, m2)
        st.pop()
        dist.pop()

    result = {}
    def runner():
        dfs(0, -1, 0, -1, -1)
        result['ans'] = list(best)

    old_limit = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old_limit, 4 * n + 100))
    threading.stack_size(256 * 1024 * 1024)
    th = threading.Thread(target=runner)
    th.start(); th.join()
    threading.stack_size(0)
    sys.setrecursionlimit(old_limit)
    return result['ans']

for e, nums_, exp in examples:
    print(longest_recursive(e, nums_), '| kỳ vọng', exp)

[9, 3] | kỳ vọng [9, 3]
[5, 2] | kỳ vọng [5, 2]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in thứ tự vào nút, độ sâu, `prev` của giá trị hiện tại, `(M1, M2)`, điểm bắt đầu `L` và ứng viên.

In [10]:
def trace_top2(edges, nums):
    n = len(nums)
    g = build(n, edges)
    last, dist, path = defaultdict(list), [], []
    best = [0, 1]
    stack = [(0, -1, 0, 0, -1, -1)]
    while stack:
        u, p, w, phase, m1, m2 = stack.pop()
        if phase == 1:
            last[nums[u]].pop(); dist.pop(); path.pop()
            continue
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w); path.append(u)
        st = last[nums[u]]
        pv = st[-1] if st else -1
        if pv > m1: m1, m2 = pv, m1
        elif pv > m2: m2 = pv
        st.append(d)
        L = m2 + 1
        cand = [dist[d] - dist[L], d - L + 1]
        mark = ''
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best, mark = cand, '  <- tốt hơn'
        print(f"nút {u} (giá trị {nums[u]}), d={d}, prev={pv:2d}, (M1,M2)=({m1:2d},{m2:2d}), "
              f"đoạn {path[L:]} -> {cand}{mark}")
        stack.append((u, p, w, 1, m1, m2))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, m1, m2))
    print('kết quả:', best)

trace_top2(*examples[0][:2])

nút 0 (giá trị 1), d=0, prev=-1, (M1,M2)=(-1,-1), đoạn [0] -> [0, 1]
nút 1 (giá trị 1), d=1, prev= 0, (M1,M2)=( 0,-1), đoạn [0, 1] -> [1, 2]  <- tốt hơn
nút 3 (giá trị 3), d=2, prev=-1, (M1,M2)=( 0,-1), đoạn [0, 1, 3] -> [2, 3]  <- tốt hơn
nút 6 (giá trị 1), d=3, prev= 1, (M1,M2)=( 1, 0), đoạn [1, 3, 6] -> [6, 3]  <- tốt hơn
nút 8 (giá trị 0), d=4, prev=-1, (M1,M2)=( 1, 0), đoạn [1, 3, 6, 8] -> [9, 4]  <- tốt hơn
nút 5 (giá trị 2), d=3, prev=-1, (M1,M2)=( 0,-1), đoạn [0, 1, 3, 5] -> [4, 4]
nút 2 (giá trị 0), d=2, prev=-1, (M1,M2)=( 0,-1), đoạn [0, 1, 2] -> [4, 3]
nút 4 (giá trị 1), d=3, prev= 1, (M1,M2)=( 1, 0), đoạn [1, 2, 4] -> [9, 3]  <- tốt hơn
nút 7 (giá trị 1), d=4, prev= 3, (M1,M2)=( 3, 1), đoạn [2, 4, 7] -> [8, 3]
kết quả: [9, 3]


Quan sát: tại nút `8` đường `0,1,3,6,8` có giá trị `1` ở các độ sâu `0, 1, 3` ⇒ `prev` lần lượt `0` (ở độ sâu 1) và `1` (ở độ sâu 3) ⇒ `M2 = 0`, `L = 1` ⇒ đoạn `[1, 3, 6, 8]`.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def longestSpecialPath(self, edges: List[List[int]], nums: List[int]) -> List[int]:
        n = len(nums)
        g = [[] for _ in range(n)]
        for u, v, w in edges:
            g[u].append((v, w))
            g[v].append((u, w))
        last = defaultdict(list)
        dist = []
        best_len, best_cnt = 0, 1
        stack = [(0, -1, 0, False, -1, -1)]
        while stack:
            u, p, w, leaving, m1, m2 = stack.pop()
            if leaving:
                last[nums[u]].pop()
                dist.pop()
                continue
            d = len(dist)
            dist.append((dist[-1] if dist else 0) + w)
            st = last[nums[u]]
            pv = st[-1] if st else -1
            if pv > m1:
                m1, m2 = pv, m1
            elif pv > m2:
                m2 = pv
            st.append(d)
            L = m2 + 1
            length, cnt = dist[d] - dist[L], d - L + 1
            if length > best_len or (length == best_len and cnt < best_cnt):
                best_len, best_cnt = length, cnt
            stack.append((u, p, w, True, m1, m2))
            for v, ww in g[u]:
                if v != p:
                    stack.append((v, u, ww, False, m1, m2))
        return [best_len, best_cnt]

for e, nums_, exp in examples:
    got = Solution().longestSpecialPath(e, nums_)
    print(got, 'OK' if got == exp else 'SAI')

[9, 3] OK
[5, 2] OK


## 10. Vì sao thuật toán đúng

1. **Mọi đường đi xuống** là đoạn `[L, d]` của đường gốc → nút cuối; DFS thăm mọi nút cuối, và tại thời điểm thăm, ngăn xếp `dist`/`last` mô tả đúng đường gốc → nút đó (push khi vào, pop khi ra).
2. **`last[v][-1]` đúng là `prev`**: ngăn xếp của `v` chỉ chứa các độ sâu trên đường hiện tại (các nhánh đã rời bị pop).
3. **`(M1, M2)` đúng**: là hai giá trị lớn nhất của `prev` trên đường, vì được truyền từ cha (đường của cha là tiền tố) và cập nhật thêm đúng một giá trị mới. Bản lưu trong ngăn xếp mang `(M1, M2)` của riêng từng nút ⇒ không bị nhánh anh em làm bẩn.
4. **`L = M2 + 1` tối ưu**: theo bổ đề mục 6, `[L, d]` hợp lệ ⇔ tối đa một `prev ≥ L` ⇔ `L > M2`.
5. **Tiêu chí phụ**: tại mỗi nút cuối, đoạn dài nhất là duy nhất (`L*`); mọi đoạn ngắn hơn có độ dài nhỏ hơn **thực sự** (cạnh ≥ 1) nên không thể hòa với đoạn dài nhất toàn cục trừ khi chính nó là `L*` của nút cuối đó. Vì thế so sánh `(độ dài lớn, số nút nhỏ)` qua các ứng viên `L*` là đủ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Leo ngược từng nút | `O(n · depth)` (tệ nhất `O(n²)`) | `O(n)` |
| 2. Cửa sổ trượt có hoàn tác | tệ nhất `O(n · depth)` | `O(n)` |
| 3. Hai `prev` lớn nhất (lặp) | `O(n)` | `O(n)` |
| 4. Như 3 nhưng đệ quy | `O(n)` | `O(n)` stack hệ thống |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'window': longest_window, 'top2': longest_top2, 'recursive': longest_recursive,
         'submit': lambda e, a: Solution().longestSpecialPath(e, a)}
edge = [
    ([[0, 1, 5]], [7, 7], [5, 2]),                       # hai nút cùng giá trị: được phép lặp một lần
    ([[0, 1, 2], [1, 2, 3]], [4, 4, 4], [3, 2]),          # ba lần cùng giá trị: chỉ lấy được 2 nút
    ([[0, 1, 2], [1, 2, 3], [2, 3, 4]], [1, 1, 2, 2], [7, 3]),  # hai cặp: bỏ một cặp
    ([[0, 1, 1], [0, 2, 1]], [1, 2, 3], [1, 2]),          # cùng độ dài: chọn ít nút (đều 2)
]
for e, a, exp in examples + edge:
    assert longest_brute(e, a) == exp, (e, a, longest_brute(e, a))
    for name, f in impls.items():
        assert f(e, a) == exp, (name, e, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (số hiệu nút xáo trộn, cạnh đảo chiều)

In [13]:
import random

def random_tree(n, vmax, wmax, rng):
    edges = [[rng.randrange(v), v, rng.randint(1, wmax)] for v in range(1, n)]
    perm = list(range(1, n)); rng.shuffle(perm); perm = [0] + perm      # giữ gốc là 0
    out = []
    for u, v, w in edges:
        u, v = perm[u], perm[v]
        out.append([u, v, w] if rng.random() < 0.5 else [v, u, w])
    return out, [rng.randint(0, vmax) for _ in range(n)]

rng = random.Random(3486)
cnt = 0
for _ in range(1500):
    n = rng.randint(2, 14)
    e, a = random_tree(n, rng.randint(0, 4), rng.choice([1, 3, 10]), rng)
    exp = longest_brute(e, a)
    for name, f in impls.items():
        assert f(e, a) == exp, (name, e, a)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 1500 cây ngẫu nhiên.


### 12.3. Ca lớn: chuỗi `5·10^4` nút, cây "chổi", cây ngẫu nhiên

In [14]:
import time
n = 50000
rng = random.Random(1)
chain = ([[i, i + 1, rng.randint(1, 1000)] for i in range(n - 1)], [rng.randint(0, 50000) for _ in range(n)])

def make_broom(n):
    # cán chổi: chuỗi half nút, giá trị phân biệt trừ hai nút cuối bằng nhau;
    # mỗi lá có giá trị trùng nút thứ (half-3) => L phải nhảy gần hết cán
    half = n // 2
    hv = list(range(half)); hv[half - 1] = hv[half - 2]
    edges = [[i, i + 1, 1] for i in range(half - 1)] + [[half - 1, j, 1] for j in range(half, n)]
    return edges, hv + [hv[half - 3]] * (n - half)

broom = make_broom(n)
rand_big = random_tree(n, 30, 1000, rng)
for name, (e, a) in [('chuỗi', chain), ('chổi', broom), ('ngẫu nhiên', rand_big)]:
    t = time.time(); r3 = Solution().longestSpecialPath(e, a); t3 = time.time() - t
    print(f"{name:10s}: {r3} | top2 (lời giải nộp) {t3:.2f}s")

chuỗi     : [463133, 932] | top2 (lời giải nộp) 0.12s
chổi      : [24999, 25000] | top2 (lời giải nộp) 0.04s
ngẫu nhiên: [10399, 18] | top2 (lời giải nộp) 0.13s


So sánh tốc độ cách 2 (cửa sổ có hoàn tác) với cách 3 trên cây chổi kích thước tăng dần — cách 2 tăng gần **bình phương**:

In [15]:
for m in [1000, 2000, 4000]:
    e, a = make_broom(m)
    t = time.time(); r3 = longest_top2(e, a); t3 = time.time() - t
    t = time.time(); r2 = longest_window(e, a); t2 = time.time() - t
    assert r2 == r3 == longest_brute(e, a) if m <= 1000 else r2 == r3
    print(f"n = {m:5d}: {r3} | top2 {t3:.3f}s | cửa sổ có hoàn tác {t2:.3f}s")
for name, (e, a) in [('chuỗi', chain), ('ngẫu nhiên', rand_big)]:
    t = time.time(); r2 = longest_window(e, a); t2 = time.time() - t
    assert r2 == Solution().longestSpecialPath(e, a)
    print(f"{name}: cửa sổ có hoàn tác {t2:.2f}s (khớp)")

n =  1000: [499, 500] | top2 0.001s | cửa sổ có hoàn tác 0.053s


n =  2000: [999, 1000] | top2 0.002s | cửa sổ có hoàn tác 0.241s


n =  4000: [1999, 2000] | top2 0.003s | cửa sổ có hoàn tác 0.859s


chuỗi: cửa sổ có hoàn tác 0.14s (khớp)


ngẫu nhiên: cửa sổ có hoàn tác 0.13s (khớp)


Cây "chổi": mỗi lá làm cửa sổ của cách 2 dịch `L` qua gần như cả cán chổi rồi hoàn tác ⇒ thời gian cách 2 tăng khoảng 4 lần khi `n` gấp đôi, còn cách 3 vẫn tuyến tính (với `n = 5·10^4` cách 2 sẽ mất hàng phút nên không chạy ở đây).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ dùng `M1` (lời giải của phiên bản "không lặp", LC 3425)

In [16]:
def wrong_only_m1(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, best = defaultdict(list), [], [0, 1]
    stack = [(0, -1, 0, 0, -1)]
    while stack:
        u, p, w, phase, m1 = stack.pop()
        if phase:
            last[nums[u]].pop(); dist.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        st = last[nums[u]]
        m1 = max(m1, st[-1] if st else -1)
        st.append(d)
        L = m1 + 1                                   # SAI: không cho phép giá trị nào lặp
        cand = [dist[d] - dist[L], d - L + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, 1, m1))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, m1))
    return best

e, a, _ = examples[0]
print('sai :', wrong_only_m1(e, a), '| đúng:', Solution().longestSpecialPath(e, a))

sai : [8, 3] | đúng: [9, 3]


**Vì sao sai:** đề cho phép **một** giá trị xuất hiện hai lần ⇒ điểm bắt đầu phải vượt `M2`, không phải `M1`.

### Lỗi 2 — Cho phép "một giá trị bị lặp" nhưng không giới hạn số lần lặp của nó

In [17]:
def wrong_one_dup_value(edges, nums):
    # Cửa sổ hợp lệ nếu số GIÁ TRỊ có cnt >= 2 là tối đa 1 (cho phép cnt = 3, 4, ...)
    n = len(nums)
    par, pw, _ = parents(n, edges)
    best = (0, -1)
    for x in range(n):
        cnt, length, k, y = Counter(), 0, 0, x
        while y != -1:
            cnt[nums[y]] += 1; k += 1
            if sum(1 for c in cnt.values() if c >= 2) > 1:
                break
            best = max(best, (length, -k))
            length += pw[y]; y = par[y]
    return [best[0], -best[1]]

e, a = [[0, 1, 2], [1, 2, 3]], [4, 4, 4]
print('sai :', wrong_one_dup_value(e, a), '| đúng:', Solution().longestSpecialPath(e, a))

sai : [5, 3] | đúng: [3, 2]


**Vì sao sai:** "xuất hiện hai lần" nghĩa là tối đa **hai** lần. Giá trị xuất hiện ba lần tạo ra hai vị trí `prev ≥ L` ⇒ phải bị loại. Đếm "số lần dư" (như cách 2) hoặc dùng `M2` (cách 3) xử lý đúng tự động.

### Lỗi 3 — Hòa độ dài thì lấy đường **nhiều** nút (hoặc giữ đường tìm thấy đầu tiên)

In [18]:
def wrong_tie(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, best = defaultdict(list), [], [0, 1]
    stack = [(0, -1, 0, 0, -1, -1)]
    while stack:
        u, p, w, phase, m1, m2 = stack.pop()
        if phase:
            last[nums[u]].pop(); dist.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        st = last[nums[u]]; pv = st[-1] if st else -1
        if pv > m1: m1, m2 = pv, m1
        elif pv > m2: m2 = pv
        st.append(d)
        L = m2 + 1
        cand = [dist[d] - dist[L], d - L + 1]
        if cand[0] > best[0]:                       # SAI: bỏ qua tiêu chí phụ khi hòa
            best = cand
        stack.append((u, p, w, 1, m1, m2))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, m1, m2))
    return best

e, a, _ = examples[0]
print('sai :', wrong_tie(e, a), '| đúng:', Solution().longestSpecialPath(e, a))

sai : [9, 4] | đúng: [9, 3]


**Vì sao sai:** ở ví dụ 1, DFS gặp đường `1→3→6→8` (4 nút) trước `1→2→4` (3 nút), cả hai dài 9. Không xét hòa thì giữ 4. Phải so sánh theo cặp `(độ dài lớn hơn, số nút nhỏ hơn)`.

### Lỗi 4 — DFS đệ quy với giới hạn mặc định trên cây dạng chuỗi

In [19]:
def naive_recursive(edges, nums):
    n = len(nums); g = build(n, edges)
    def dfs(u, p, depth):
        res = depth
        for v, _ in g[u]:
            if v != p:
                res = max(res, dfs(v, u, depth + 1))
        return res
    return dfs(0, -1, 0)

try:
    naive_recursive(*chain)
except RecursionError as ex:
    print('RecursionError:', ex)

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** chuỗi `5·10^4` nút cần độ sâu đệ quy `5·10^4` > giới hạn mặc định `1000`. Dùng DFS lặp (cách 3) hoặc nâng giới hạn + stack thread lớn (cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi xuống từ tổ tiên tới hậu duệ" + điều kiện kiểu cửa sổ ⇒ coi đường gốc → nút là **mảng động**, duyệt DFS, push/pop.
- Điều kiện "không lặp" ⇒ `L = max(prev) + 1`; "lặp tối đa một lần" ⇒ `L = (max thứ hai của prev) + 1`. Tổng quát "tối đa `t` vị trí vi phạm" ⇒ giữ `t + 1` giá trị lớn nhất.
- Thông tin truyền **xuống** con qua tham số (như `M1, M2`) thì tự hoàn tác; thông tin dùng chung (`last`, `dist`) phải pop thủ công.
- Cây `≥ 10^4` nút trong Python ⇒ DFS lặp.

## 15. Biến thể của bài

1. **LC 3425 — Longest Special Path:** không cho lặp ⇒ chỉ cần `M1`.
2. Cho phép tối đa `t` lần lặp dư ⇒ giữ `t + 1` giá trị `prev` lớn nhất (danh sách nhỏ đã sắp).
3. Đếm số đường đặc biệt (thay vì dài nhất) ⇒ cộng `d − L + 1` tại mỗi nút.
4. Trên mảng: "đoạn con dài nhất có tối đa một giá trị lặp" — bản tuyến tính của cùng ý tưởng.

In [20]:
def count_special_paths(edges, nums):
    # Biến thể 3: số đường đặc biệt (kể cả đường 1 nút) = Σ (d - L + 1)
    n = len(nums); g = build(n, edges)
    last, depth_stack, total = defaultdict(list), [0], 0
    stack = [(0, -1, 0, -1, -1)]
    path_len = 0
    while stack:
        u, p, phase, m1, m2 = stack.pop()
        if phase:
            last[nums[u]].pop(); path_len -= 1; continue
        d = path_len; path_len += 1
        st = last[nums[u]]; pv = st[-1] if st else -1
        if pv > m1: m1, m2 = pv, m1
        elif pv > m2: m2 = pv
        st.append(d)
        total += d - m2
        stack.append((u, p, 1, m1, m2))
        for v, _ in g[u]:
            if v != p:
                stack.append((v, u, 0, m1, m2))
    return total

def count_special_brute(edges, nums):
    n = len(nums); par, pw, _ = parents(n, edges); total = 0
    for x in range(n):
        cnt, extra, y = Counter(), 0, x
        while y != -1:
            cnt[nums[y]] += 1
            extra += cnt[nums[y]] >= 2
            if extra > 1: break
            total += 1; y = par[y]
    return total

rng = random.Random(15)
for _ in range(300):
    e, a = random_tree(rng.randint(2, 12), 3, 5, rng)
    assert count_special_paths(e, a) == count_special_brute(e, a)
print('Biến thể 3 khớp vét cạn; ví dụ 1 có', count_special_paths(*examples[0][:2]), 'đường đặc biệt.')

Biến thể 3 khớp vét cạn; ví dụ 1 có 26 đường đặc biệt.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Leo ngược + Counter | `O(n · depth)` | Oracle |
| 2 | Cửa sổ trượt có hoàn tác | tệ nhất `O(n · depth)` | Dễ nghĩ, chậm trên cây chổi |
| 3 | Hai `prev` lớn nhất, DFS lặp | `O(n)` | **Nộp bài** |
| 4 | Như 3, đệ quy + thread | `O(n)` | Gọn nhưng phụ thuộc stack |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `edges = [[0,1,4],[1,2,4]]`, `nums = [2,3,2]`.

**Bài 2 (tính tay):** cây chuỗi `0-1-2-3-4` mỗi cạnh dài 1, `nums = [5,5,6,6,7]`.

**Bài 3 (điền TODO):** viết `longest_no_repeat(edges, nums)` — phiên bản không cho lặp (LC 3425).

**Bài 4 (tự viết lại):** cho phép tối đa `t` lần lặp dư — tổng quát hóa cách 3.

**Bài 5 (mini challenge):** trên **mảng** `a` (không phải cây) với trọng số `w[i]` giữa `a[i]` và `a[i+1]`, tìm đoạn có tổng trọng số lớn nhất mà tối đa một giá trị lặp đúng một lần.

In [21]:
def longest_no_repeat_todo(edges, nums):
    # TODO
    raise NotImplementedError

def longest_t_repeats_todo(edges, nums, t):
    # TODO
    raise NotImplementedError

def array_version_todo(a, w):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Giá trị 2 lặp ở nút 0 và 2. (2) Một lần lặp được phép. (3) Cả đường `0→1→2` hợp lệ.

**Bài 2.** (1) Hai cặp lặp (5 và 6). (2) Phải bỏ một cặp. (3) Đoạn `1,2,3,4` hoặc `0,1,2` …: so độ dài và số nút.

**Bài 3.** (1) Chỉ cần `M1`. (2) `L = M1 + 1`. (3) Đúng như lỗi 1 — ở đây nó lại là lời giải đúng.

**Bài 4.** (1) Giữ danh sách `t + 1` giá trị `prev` lớn nhất (sắp giảm). (2) `L = (phần tử thứ t+1) + 1`. (3) Đối chiếu vét cạn đếm "lần dư".

**Bài 5.** (1) Mảng là cây dạng chuỗi. (2) Tái sử dụng hàm cây. (3) Hoặc cửa sổ trượt thường (không cần hoàn tác).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
a1 = Solution().longestSpecialPath([[0, 1, 4], [1, 2, 4]], [2, 3, 2])
chain5 = [[i, i + 1, 1] for i in range(4)]
a2 = Solution().longestSpecialPath(chain5, [5, 5, 6, 6, 7])
assert a1 == [8, 3] == longest_brute([[0, 1, 4], [1, 2, 4]], [2, 3, 2])
assert a2 == longest_brute(chain5, [5, 5, 6, 6, 7]) == [3, 4]
print('Bài 1:', a1, '| Bài 2:', a2, '(đoạn 1,2,3,4: giá trị 5,6,6,7)')

Bài 1: [8, 3] | Bài 2: [3, 4] (đoạn 1,2,3,4: giá trị 5,6,6,7)


In [23]:
# Bài 3, 4
import heapq

def longest_t_repeats(edges, nums, t):
    n = len(nums); g = build(n, edges)
    last, dist, best = defaultdict(list), [], [0, 1]
    stack = [(0, -1, 0, 0, tuple([-1] * (t + 1)))]
    while stack:
        u, p, w, phase, top = stack.pop()
        if phase:
            last[nums[u]].pop(); dist.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        st = last[nums[u]]; pv = st[-1] if st else -1
        top = tuple(sorted(top + (pv,), reverse=True)[:t + 1])
        st.append(d)
        L = top[t] + 1
        cand = [dist[d] - dist[L], d - L + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, 1, top))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, 0, top))
    return best

def brute_t(edges, nums, t):
    n = len(nums); par, pw, _ = parents(n, edges); best = (0, -1)
    for x in range(n):
        cnt, length, k, extra, y = Counter(), 0, 0, 0, x
        while y != -1:
            cnt[nums[y]] += 1; k += 1
            extra += cnt[nums[y]] >= 2
            if extra > t: break
            best = max(best, (length, -k)); length += pw[y]; y = par[y]
    return [best[0], -best[1]]

longest_no_repeat = lambda e, a: longest_t_repeats(e, a, 0)
rng = random.Random(19)
for _ in range(400):
    e, a = random_tree(rng.randint(2, 12), 3, 6, rng)
    for t in range(3):
        assert longest_t_repeats(e, a, t) == brute_t(e, a, t)
    assert longest_t_repeats(e, a, 1) == Solution().longestSpecialPath(e, a)
print('Bài 3 (t=0) và Bài 4 (t tổng quát) khớp vét cạn. Ví dụ 1 với t=0:', longest_no_repeat(*examples[0][:2]))

Bài 3 (t=0) và Bài 4 (t tổng quát) khớp vét cạn. Ví dụ 1 với t=0: [8, 3]


In [24]:
# Bài 5 — mảng: cửa sổ trượt thường (không cần hoàn tác vì không quay lui)
def array_version(a, w):
    pre = [0]
    for x in w:
        pre.append(pre[-1] + x)
    cnt, extra, L, best = Counter(), 0, 0, (0, -1)
    for d, x in enumerate(a):
        cnt[x] += 1
        extra += cnt[x] >= 2
        while extra > 1:
            y = a[L]
            extra -= cnt[y] >= 2
            cnt[y] -= 1
            L += 1
        best = max(best, (pre[d] - pre[L], -(d - L + 1)))
    return [best[0], -best[1]]

rng = random.Random(5)
for _ in range(300):
    m = rng.randint(2, 12)
    a = [rng.randint(0, 3) for _ in range(m)]
    w = [rng.randint(1, 5) for _ in range(m - 1)]
    edges = [[i, i + 1, w[i]] for i in range(m - 1)]
    assert array_version(a, w) == Solution().longestSpecialPath(edges, a)
print('Bài 5 khớp với lời giải cây trên chuỗi.', array_version([1, 2, 1, 3, 1], [2, 2, 2, 2]))

Bài 5 khớp với lời giải cây trên chuỗi. [6, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
last = defaultdict(list); dist = []; stack = [(0, -1, 0, False, -1, -1)]
while stack:
    u, p, w, leaving, m1, m2 = stack.pop()
    if leaving: last[nums[u]].pop(); dist.pop(); continue
    d = len(dist); dist.append((dist[-1] if dist else 0) + w)
    pv = last[nums[u]][-1] if last[nums[u]] else -1
    if pv > m1: m1, m2 = pv, m1
    elif pv > m2: m2 = pv
    last[nums[u]].append(d)
    L = m2 + 1   # cập nhật (dist[d] - dist[L], d - L + 1)
    stack.append((u, p, w, True, m1, m2)); push các con với (m1, m2)
```

**Checklist:**

- [ ] Dùng `M2` (lớn thứ hai), không phải `M1`?
- [ ] Giá trị lặp 3 lần bị loại?
- [ ] Hòa độ dài ⇒ chọn **ít nút** hơn?
- [ ] Đường một nút: `[0, 1]` là giá trị khởi tạo?
- [ ] DFS lặp, pop `last` và `dist` khi rời nút?